<a href="https://colab.research.google.com/github/scottaudrey/svd-food-recommendation-system/blob/main/notebooks/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Part A: Load the Ratings

In [11]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

In [12]:
url = "https://github.com/ryannewhouse11/FoodReviewData/releases/latest/download/recipereview.zip"
urllib.request.urlretrieve(url, "/content/recipereview.zip")

with zipfile.ZipFile("/content/recipereview.zip") as z:
  csv_files = [
      name for name in z.namelist()
      if name.lower().endswith(".csv") and not name.startswith("_MACOSX/")
  ]


  with z.open(csv_files[0]) as f:
    ratings = pd.read_csv(
        f,
        sep=",",
        header=0,
        names=["num_records", "recipe_number", "recipe_code", "recipe_name",
               "comment_id", "user_id", "user_name", "user_reputation", "created_at",
               "reply_count", "thumbs_up", "thumbs_down", "stars", "best_score", "text"]
    )
  print(ratings.head())
  print("Number of ratings:", len(ratings))


   num_records  recipe_number  recipe_code         recipe_name  \
0            0              1        14299  Creamy White Chili   
1            1              1        14299  Creamy White Chili   
2            2              1        14299  Creamy White Chili   
3            3              1        14299  Creamy White Chili   
4            4              1        14299  Creamy White Chili   

                                        comment_id         user_id  \
0  sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM  u_9iFLIhMa8QaG   
1  sp_aUSaElGf_14299_c_2FsPC83HtzCsQAtOxlbL6RcaPbY  u_Lu6p25tmE77j   
2  sp_aUSaElGf_14299_c_2FPrSGyTv7PQkZq37j92r9mYGkP  u_s0LwgpZ8Jsqq   
3  sp_aUSaElGf_14299_c_2DzdSIgV9qNiuBaLoZ7JQaartoC  u_fqrybAdYjgjG   
4  sp_aUSaElGf_14299_c_2DtZJuRQYeTFwXBoZRfRhBPEXjI  u_XXWKwVhKZD69   

    user_name  user_reputation  created_at  reply_count  thumbs_up  \
0     Jeri326                1  1665619889            0          0   
1     Mark467               50  1665277687

# Part B: Make an honest train/test split

In [13]:
def split_ratings(ratings, test_frac=0.10, seed=0):
  rng = np.random.default_rng(seed)
  is_test = np.zeros(len(ratings), dtype=bool)

  user_left = ratings.groupby("user_id").size().to_dict()
  item_left = ratings.groupby("recipe_code").size().to_dict()

  for pos in rng.permutation(len(ratings)):  # goes through the ratings randomly
    row = ratings.iloc[pos]
    user_id, recipe_code = row["user_id"], row["recipe_code"]

    if (rng.random() < test_frac   # each of the ratings now has a 10% chance of being the test rating
      and user_left[user_id] > 1
      and item_left[recipe_code] > 1):
      is_test[pos] = True
      user_left[user_id] -= 1
      item_left[recipe_code] -= 1

  return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()  # returns the integer

train, test = split_ratings(ratings)
print("train: ", len(train), "test: ", len(test))
print("test fraction: ", round(len(test) / len(ratings), 3))

train:  17543 test:  639
test fraction:  0.035


#Part C: Build the training Matrix

In [17]:
user_ids = pd.Index(train["user_id"].unique())
recipe_ids = pd.Index(train["recipe_code"].unique())
N_USERS, N_ITEMS = len(user_ids), len(recipe_ids)

def to_matrix(df):
    R = np.full((N_USERS, N_ITEMS), np.nan)
    u = user_ids.get_indexer(df["user_id"])
    i = recipe_ids.get_indexer(df["recipe_code"])
    R[u, i] = df["stars"].to_numpy()
    return R

R = to_matrix(train)
known = ~np.isnan(R)
print("Matrix shape:", R.shape)

Matrix shape: (13812, 100)


In [21]:
truth = test["stars"].to_numpy()
u = user_ids.get_indexer(test["user_id"])
i = recipe_ids.get_indexer(test["recipe_code"])

#Part D: Score predictions with RMSE

In [25]:
def rmse(pred, truth):
  ## Root mean squared error, measured in starts.
  return np.sqrt(np.mean((pred - truth)**2))

#Part E: Build simiple Baselines

In [29]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0
item_mean[has_rating] = (
    np.nansum(R, axis=0)[has_rating] / n_item[has_rating])

print("Overall Average: ", round(mu, 3))

Overall Average:  4.283


In [31]:
scores = {}
scores["always 3"] = rmse(np.full(len(test), 3.0), truth)
scores["overall average"] = rmse(np.full(len(test), mu), truth)
scores["user average"] = rmse(user_mean[u], truth)
scores["recipe average"] = rmse(item_mean[i], truth)

for name, score in scores.items():
  print(f"{name:16s} {score:.4f}")

always 3         1.9505
overall average  1.2095
user average     1.2049
recipe average   1.1966


## Part F: Combine user and recipe effects

In [22]:
def bias_baseline(R):
    """Overall mean plus one user effect and one recipe effect."""
    known = ~np.isnan(R)
    mu = np.nanmean(R)
    b_user = np.nanmean(R - mu, axis=1)
    residual = R - mu - b_user[:, None]
    n_item = known.sum(axis=0)
    b_item = np.zeros(R.shape[1])
    has_rating = n_item > 0
    b_item[has_rating] = (
        np.nansum(residual, axis=0)[has_rating] / n_item[has_rating])
    return mu, b_user, b_item

In [32]:
mu, b_user, b_item = bias_baseline(R)
pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)
scores['user + recipe effects'] = rmse(pred, truth)
print(pd.Series(scores, name='test RMSE').round(4).sort_values())

user + recipe effects    1.1617
recipe average           1.1966
user average             1.2049
overall average          1.2095
always 3                 1.9505
Name: test RMSE, dtype: float64
